In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *#

In [0]:
#dimension delta table(target table)

customer_dim_data = [

(1,'manish','arwal','india','N','2022-09-15','2022-09-25'),
(2,'vikash','patna','india','Y','2023-08-12',None),
(3,'nikita','delhi','india','Y','2023-09-10',None),
(4,'rakesh','jaipur','india','Y','2023-06-10',None),
(5,'ayush','NY','USA','Y','2023-06-10',None),
(1,'manish','gurgaon','india','Y','2022-09-25',None),
]

customer_schema= ['id','name','city','country','active','effective_start_date','effective_end_date']

customer_dim_df = spark.createDataFrame(data= customer_dim_data,schema=customer_schema)

customer_dim_df.show()
#customer_dim_df.printSchema()


In [0]:
#source data (comming on next day)

sales_data = [

(1,1,'manish','2023-01-16','gurgaon','india',380),
(77,1,'manish','2023-03-11','bangalore','india',300),
(12,3,'nikita','2023-09-20','delhi','india',127),
(54,4,'rakesh','2023-08-10','jaipur','india',321),
(65,5,'ayush','2023-09-07','mosco','russia',765),
(89,6,'rajat','2023-08-10','jaipur','india',321)
]

sales_schema = ['sales_id', 'customer_id','customer_name', 'sales_date', 'food_delivery_address','food_delivery_country', 'food_cost']

sales_df = spark.createDataFrame(data=sales_data,schema=sales_schema)
sales_df.show()
sales_df.printSchema()

In [0]:
#Join both the df to identify the change in address.

joined_data = customer_dim_df.join(sales_df,customer_dim_df["id"]==sales_df["customer_id"],"left")
display(joined_data)

In [0]:
#identify the changes in the address and create a new record
# .where((col("food_delivery_address") != col("city")) & (col("active") == "Y")) whose food_delivery_address is not equal to city and active is Y
#.withColumn("effective_start_date", col("sales_date")) whose effective_start_date is sales_date
#.withColumn("effective_end_date", lit(None).cast("string")) whose effective_end_date is null

new_record_df = (
    joined_data
    .where((col("food_delivery_address") != col("city")) & (col("active") == "Y"))
    .withColumn("active", lit("Y"))
    .withColumn("effective_start_date", col("sales_date"))
    .withColumn("effective_end_date", lit(None).cast("string"))
    .select(
        col("customer_id").alias("id"),
        col("customer_name").alias("name"),
        col("food_delivery_address").alias("city"),
        col("food_delivery_country").alias("country"),
        "active",
        "effective_start_date",
        "effective_end_date"
    )
)
display(new_record_df)




In [0]:
# to inactive old record
# .where((col("food_delivery_address") != col("city")) & (col("active") == "Y")) whose food_delivery_address is not equal to city and active is Y
#.withColumn("effective_end_date", col("sales_date")) whose effective_end_date is sales_date
#.withColumn("active", lit("N"))


old_record = (
    joined_data
    .where((col("food_delivery_address") != col("city")) & (col("active") == "Y"))
    .withColumn("active", lit("N"))
    .withColumn("effective_end_date", col("sales_date"))
    .select(
        col("customer_id").alias("id"),
        col("customer_name").alias("name"),
        col("city"),
        col("food_delivery_country").alias("country"),
        col("active"),
        col("effective_start_date"),
        col("effective_end_date")
    )
)

display(old_record)

In [0]:
#find the new record and insert them.

#find the new record and insert them.

new_customer = (
    sales_df.join(customer_dim_df, sales_df["customer_id"] == customer_dim_df["id"], "left_anti")
    .withColumn("active", lit("Y"))
    .withColumn("effective_start_date", col("sales_date"))
    .withColumn("effective_end_date", lit(None).cast("string"))
    .select(
        col("customer_id").alias("id"),
        col("customer_name").alias("name"),
        col("food_delivery_address").alias("city"),
        col("food_delivery_country").alias("country"),
        "active",
        "effective_start_date",
        "effective_end_date"
    )
)
display(new_customer)

In [0]:
#Merge All record in One Data frame

final_record = customer_dim_df.unionByName(new_record_df).unionByName(old_record).unionByName(new_customer)
display(final_record)
